# Assessing the cheaseBS box-edge runs

Reads back every run already in `runs/`, grouped by discharge and by solver
tolerance, and asks one physics question: **did scaling the profiles actually
move the equilibrium, and did it move the way it should?**

The run notebook (`reshape_convergence.ipynb`) produces; this one only reads, so
nothing here costs solver time.

**What is and is not on this machine.** Each run directory keeps its summary
JSON, which is committed. The per-point solve artifacts -- EQDSKs, profiles,
`iteration_errors.png` -- are gitignored and exist only where the solves ran.
Every table below works anywhere; the `DischargePhysics` overlays need that
machine.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import pathlib, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "pedestal_scan.py").exists())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "reshape_convergence"))

from pedestal_scan import Campaign
import assess_helpers as ah

runs = ah.load_runs()
print(f"{len(runs)} runs, shots {sorted({r['shot'] for r in runs})}, "
      f"tolerances {sorted({r['tol_q'] for r in runs})}")
ah.inventory(runs)

## What was run

`iters` is the four points in order: `Te` 0.70, `Te` 1.30, `ne` 0.70, `ne` 1.30.
`capped` counts points that exhausted `max_iter` -- those did not converge, they
ran out. `has_deltas` marks runs solved after the whole-profile diagnostics were
added, so older runs are missing `dq_max` / `dp_max`.

In [ ]:
df = ah.frame(runs)
print(f"{len(df)} solves total")
ah.artifacts_available(runs)

## Per discharge and tolerance

Each discharge's four box-edge points, grouped by the tolerance they were solved
at. 132543 is the only shot solved at more than one tolerance.

In [ ]:
for shot in sorted(df["shot"].unique()):
    for tol in sorted(df[df.shot == shot]["tol_q"].unique()):
        sub = df[(df.shot == shot) & (df.tol_q == tol)]
        print(f"\n=== {shot}   tol_q = {tol:g}   ({sub['run'].nunique()} run(s)) ===")
        show = sub[["run", "axis", "scale", "d_ped_top", "iters", "capped",
                    "converged", "accepted", "ip_err", "q_err_x0",
                    "dq_max", "dp_max", "wall_s"]]
        display(show.style.format(
            {"d_ped_top": "{:+.1%}", "ip_err": "{:.3%}", "q_err_x0": "{:.2%}",
             "dq_max": "{:.2%}", "dp_max": "{:.2%}", "wall_s": "{:.0f}"},
            na_rep="--").hide(axis="index"))

## Iteration count: what actually sets it

`bs_change` never gates -- it sits below $10^{-4}$ on every point. `q_change`
does. Only the points whose pressure change perturbs $q$ enough to exceed
`tol_q` keep iterating, which is why the same 132543 point costs 18 iterations
at $10^{-4}$ and 2 at $10^{-3}$ **and lands in the same place**.

In [ ]:
piv = df.pivot_table(index=["shot", "tol_q"], columns=["var", "scale"],
                     values="iters", aggfunc="max", dropna=False)
display(piv)

ok = df[df.iters.notna()]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for d, m, c in (("down", "o", "tab:blue"), ("up", "^", "tab:red")):
    s = ok[ok.dir == d]
    ax[0].scatter(s.d_ped_top.abs() * 100, s.iters, marker=m, c=c, s=70,
                  label=f"scale {d}", zorder=3)
ax[0].set_xlabel("|measured pedestal-top change| (%)")
ax[0].set_ylabel("iterations")
ax[0].set_title("magnitude does not order the cost")
ax[0].legend(); ax[0].grid(alpha=.3)

for d, c in (("down", "tab:blue"), ("up", "tab:red")):
    s = ok[ok.dir == d]
    ax[1].scatter(s.ip_err * 100, s.iters, c=c, s=70, label=f"scale {d}", zorder=3)
ax[1].axvline(1.5, ls="--", c="k", lw=1)
ax[1].set_xlabel("final $I_p$ error (%)"); ax[1].set_ylabel("iterations")
ax[1].set_title("dashed = standing 1.5% offset")
ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout()

## Did the equilibrium move? The whole-profile deltas

`dq_max` / `dp_max` compare the reconstructed EQDSK against the source over the
**whole** profile, with the radius where the change peaks. This is the direct
test -- cheaseBS reshapes all of it, so a number read at two analysis radii
cannot answer the question.

Watch for two things: whether different perturbations give the *same* delta
(which would mean the number is a reconstruction offset, not a response), and
whether `dp_at` lands in the pedestal, where the axes act, or in the core.

In [ ]:
d = df[df.dq_max.notna()]
if len(d) == 0:
    print("no run has whole-profile deltas yet -- re-solve to populate them")
else:
    display(d[["shot", "tol_q", "axis", "scale", "d_ped_top",
               "dq_max", "dq_at", "dp_max", "dp_at", "ip_err"]].style.format(
        {"d_ped_top": "{:+.1%}", "dq_max": "{:.3%}", "dq_at": "{:.3f}",
         "dp_max": "{:.3%}", "dp_at": "{:.3f}", "ip_err": "{:.3%}"},
        na_rep="--").hide(axis="index"))
    print("\nIf two rows with different perturbations share dq_max/dp_max to "
          "several digits, neither moved the equilibrium -- what is being "
          "measured is the reconstruction's own offset, not a response.")

## Does $I_p$ track the pressure change?

With `qspec=on` CHEASE imposes $q$ and $I_p$ is an output, responding to the
profiles only through the bootstrap current. So $I_p$ error against the source
target is a proxy for whether the reshape reached the current profile at all.

A physical response is monotonic in the pedestal-top change and roughly
symmetric about it. Anything one-sided is a finding.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for shot, m in zip(sorted(df.shot.unique()), ["o", "s", "^", "D"]):
    s = df[(df.shot == shot) & df.ip_err.notna()]
    for var, open_face in (("Te", True), ("ne", False)):
        t = s[s["var"] == var]
        if open_face:
            ax.scatter(t.d_ped_top * 100, t.ip_err * 100, marker=m, s=80,
                       facecolors="none", edgecolors="C0", zorder=3,
                       label=f"{shot} Te")
        else:
            ax.scatter(t.d_ped_top * 100, t.ip_err * 100, marker=m, s=80,
                       c="C3", zorder=3, label=f"{shot} ne")
ax.axhline(1.5, ls="--", c="k", lw=1)
ax.axvline(0, ls=":", c="grey", lw=1)
ax.set_xlabel("measured pedestal-top change (%)")
ax.set_ylabel("final $I_p$ error (%)")
ax.set_title("$I_p$ response to the reshape (open = Te, filled = ne)")
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=.3)
fig.tight_layout()
print("Points on the dashed line did not move the current profile.")

## Where the response is established: the per-iteration trace

cheaseBS prints `Ip`, `rel_error`, `bs_change` and `q_change` every iteration,
but the summary JSON keeps only the final scalars -- so the trace survives *only*
in the run notebook's stored cell output. It is the direct evidence for what
stops the loop, and nothing else in the repo carries it.

Two things to read off it:

1. **`Ip` at iteration 00**, before any bootstrap feedback. If the points already
   differ there, the equilibrium response is set by the pressure change itself
   and the loop is not what produces it.
2. **`q_change` vs `bs_change` at iteration 01** against `tol_q` / `tol_bs`.
   Whichever one sits above tolerance is what keeps the loop running.

In [ ]:
tr = ah.iteration_trace(ROOT / "reshape_convergence" / "reshape_convergence.ipynb")
if tr.empty:
    print("no executed run notebook found -- re-run reshape_convergence.ipynb "
          "and keep its output to populate this")
else:
    run_tol = {r["tag"]: r["tol_q"] for r in runs}
    tol = next((v for k, v in run_tol.items() if k in set(tr.run.dropna())), None)
    print(f"trace from run(s): {sorted(set(tr.run.dropna()))}  tol_q={tol}")
    display(tr.style.format({"ip_a": "{:,.0f}", "rel_err": "{:.4%}",
                             "bs_change": "{:.2e}", "q_change": "{:.2e}"},
                            na_rep="--").hide(axis="index"))

    it0 = tr[tr["iter"] == 0]
    it1 = tr[tr["iter"] == 1]
    print("\niteration 00 -- before any feedback:")
    for _, r in it0.iterrows():
        print(f"  {r.axis:<14} {r.scale:.2f}  Ip = {r.ip_a:,.0f} A   "
              f"rel_err = {r.rel_err:.3%}")
    print(f"\nspread at iteration 00: {it0.ip_a.max() - it0.ip_a.min():,.0f} A")
    print(f"moved by iteration 01:   "
          f"{(it1.ip_a.values - it0.ip_a.values).mean():,.0f} A (mean)")
    if tol:
        print(f"\nagainst tol_q = {tol:g}:")
        for _, r in it1.iterrows():
            flag = "OVER -> keeps iterating" if r.q_change > tol else "under -> stops"
            print(f"  {r.axis:<14} {r.scale:.2f}  q_change = {r.q_change:.2e}  {flag}")

## The symmetry test

Grad-Shafranov and the Sauter/Wesson bootstrap are **linear operators**, so to
first order

$$\delta I_p(+f) = -\,\delta I_p(-f) + \mathcal{O}(f^2)$$

These runs have no `scale = 1.0` control, so $I_p(0)$ is unknown — but it is the
*same number for every axis of a discharge*. Decompose each axis's two points:

$$\text{odd} = \tfrac{1}{2}\left[I_p(\text{up}) - I_p(\text{down})\right]
\qquad
\text{mid} = \tfrac{1}{2}\left[I_p(\text{up}) + I_p(\text{down})\right] = I_p(0) + \text{even}$$

Under a linear response every axis of one discharge must share one `mid`.
**So the spread in `mid` across axes measures the rectification directly, with
no baseline assumed.** `rect_A` is each axis's displacement from the smallest
`mid` in that discharge; `rect_over_odd` is that displacement as a fraction of
the linear response.

A purely linear, symmetric response gives `rect_over_odd = 0`.

In [ ]:
sym = ah.symmetry(df)
display(sym.style.format(
    {"Ip_up_A": "{:,.0f}", "Ip_down_A": "{:,.0f}", "d_ped_top": "{:.1%}",
     "odd_A": "{:,.0f}", "sens_A_per_unit": "{:,.0f}", "mid_A": "{:,.0f}",
     "rect_A": "{:,.0f}", "rect_over_odd": "{:.2f}"},
    na_rep="--").hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 4))
lbl = [f"{r.shot}\n{r.axis.split('_')[0]} tol={r.tol_q:g}" for r in sym.itertuples()]
x = np.arange(len(sym))
ax.bar(x - .2, sym.odd_A, .4, label="odd (linear response)", color="tab:blue")
ax.bar(x + .2, sym.rect_A, .4, label="even (rectification)", color="tab:red")
ax.set_xticks(x); ax.set_xticklabels(lbl, fontsize=7)
ax.set_ylabel("$I_p$ (A)")
ax.set_title("A linear response has no red bar")
ax.legend(); ax.grid(alpha=.3, axis="y")
fig.tight_layout()

## Sensitivity against the bootstrap prediction

Normalising the linear response by the pedestal-top change the axis actually
bought gives a sensitivity, $\text{odd}/|f|$, in amps per unit fraction. That
is the quantity theory predicts.

In the banana regime (Wesson),

$$j_\text{BS} = -\frac{\varepsilon^{1/2}}{B_\theta}
\left[2.44(T_e{+}T_i)\frac{dn}{dr} + 0.69\,n\frac{dT_e}{dr}
- 1.54\,n\frac{dT_i}{dr}\right]$$

Every term is linear in $n$, so an `ne` scan multiplies the whole bracket:
$\delta j_\text{BS}/j_\text{BS} = f$ exactly. A `Te` scan touches only the
first two terms and leaves $T_i$ alone, so it moves roughly half as much.

$$\boxed{\;\text{sens}(n_e)\,/\,\text{sens}(T_e)\;\approx\;2\;}$$

In [ ]:
p = sym.pivot_table(index=["shot", "tol_q"], columns="axis",
                    values="sens_A_per_unit")
p.columns = [c.split("_")[0] for c in p.columns]
p["ne/Te"] = p["ne"] / p["Te"]
display(p.style.format({"Te": "{:,.0f}", "ne": "{:,.0f}", "ne/Te": "{:.1f}"}))

print("Bootstrap prediction: ne/Te ~ 2.")
print("ne sensitivity should also be roughly discharge-independent -- it is the "
      "same physics on four NSTX pedestals.")

fig, ax = plt.subplots(figsize=(7, 4))
for axis, c in (("Te", "tab:orange"), ("ne", "tab:blue")):
    ax.scatter(range(len(p)), p[axis], s=90, c=c, label=axis, zorder=3)
ax.set_xticks(range(len(p)))
ax.set_xticklabels([f"{a}\ntol={b:g}" for a, b in p.index], fontsize=8)
ax.set_ylabel(r"$dI_p\,/\,d(\Delta$ ped top$)$   (A per unit fraction)")
ax.set_yscale("log"); ax.grid(alpha=.3, which="both")
ax.set_title("Response per unit pedestal-top change")
ax.legend(); fig.tight_layout()

## Profiles and equilibrium overlaid

`DischargePhysics.plot()` draws geometry, $T$ and $n$ profiles, and the
$q$, $F$, $p$, $p'$, $FF'$ panels in one figure, so a scaling change and the
equilibrium change it caused are read together.

Split by axis on purpose: `Te` and `ne` separately is three curves rather than
five, and within a scan the source is the control. **The `Te` scan should leave
$n_e$ untouched and the `ne` scan should leave $T_e$ untouched** -- if either
moves, the transform is not doing what the axis name says.

In [ ]:
RUN = runs[-1]          # newest; pick another with runs[i] or by tag
camp = Campaign([RUN["shot"]])
print(RUN["tag"], "shot", RUN["shot"], "tol_q", RUN["tol_q"])
fig = ah.overlay(camp, RUN, var="Te")

In [ ]:
fig = ah.overlay(camp, RUN, var="ne")

In [ ]:
# All four at once, for the equilibrium panels where the spread is the point.
fig = ah.overlay(camp, RUN)

## Cross-tolerance check on 132543

The same discharge solved at $10^{-4}$ and $10^{-3}$. If the two tolerances give
the same equilibrium, the extra iterations bought nothing and `tol_q` is a cost
knob rather than a quality knob.

In [ ]:
t = df[df.shot == 132543].sort_values(["tol_q", "axis", "scale"])
display(t[["run", "tol_q", "axis", "scale", "iters", "ip_err", "q_err_x0",
           "wall_s"]].style.format(
    {"ip_err": "{:.4%}", "q_err_x0": "{:.3%}", "wall_s": "{:.0f}"},
    na_rep="--").hide(axis="index"))

print("\nSame points, two tolerances:")
display(t.pivot_table(index=["axis", "scale"], columns="tol_q",
                      values=["iters", "ip_err", "q_err_x0"], aggfunc="first"))

## Facts, and what is still unverified

**Established by more than one independent route:**

- The `ne` up-scan moves the equilibrium; the `ne` down-scan does not. Seen in
  the final $I_p$, in the iteration-00 $I_p$, and in the whole-profile
  `dp_max` / `dq_max`.
- Iteration count is set by `tol_q` acting on `q_change`. `bs_change` is below
  $10^{-4}$ on every point and never gates.
- $q$ and $\hat{s}$ barely move. **This is expected**: `qspec=on` imposes
  $q(\psi)$, so the response is forced into $FF'$, $j_\phi$ and the flux-surface
  geometry, not into $q$.
- Three of four 132543 points share `dq_max` and `dp_max` to four significant
  figures despite different perturbations, and both peak away from the pedestal
  ($\rho_t = 0.98$ in $q$, $0.42$ in $p$). That is a reconstruction offset, not
  a response.

**Not established — do not quote these as settled:**

- The floored $p_\text{fast}$ split as the *cause*. It has the right shape and
  nothing else in the chain does, but it has not been demonstrated. The check is
  the Step-1 thermal/fast split across the four baseline directories.
- Whether 129038 is divergent or merely slow. All four points hit `max_iter`.
- The reconstruction $T_e$ profiles read higher than `Discharge.scaled()`
  produces on 132543. Unexplained; needs the artifacts.
- Nothing here characterises `bootstrap_mix` / `istar_mix`, `max_iter`, or the
  width axes — none has been varied.

**Missing control that would settle several of these at once:** a `scale = 1.0`
point per discharge. It pins $I_p(0)$, turning the relative rectification above
into an absolute one, and separates the reconstruction's standing offset from
the reshape response. One extra solve per discharge.